In [0]:
import pandas as pd
data = [
    (1, 'Alice', 'Sales', 60000, '2020-03-01', 5),
    (2, 'Bob', 'Sales', 55000, '2021-07-15', 4),
    (3, 'Charlie', 'HR', 50000, '2019-02-10', 3),
    (4, 'Diana', 'HR', 52000, '2020-08-23', 5),
    (5, 'Evan', 'IT', 70000, '2018-06-12', 2),
    (6, 'Fiona', 'IT', 72000, '2019-11-30', 4),
    (7, 'George', 'IT', 68000, '2021-04-18', 5),
    (8, 'Hannah', 'Marketing', 48000, '2020-01-20', 3),
    (9, 'Ian', 'Marketing', 50000, '2021-05-22', 4),
    (10, 'Jane', 'Sales', 58000, '2019-12-01', 3)
]
columns = [
    "EmpID",
    "EmpName",
    "Department",
    "Salary",
    "JoiningDate",
    "PerformanceRating"
]
df=pd.DataFrame(data, columns=columns)

In [0]:
display(df)

In [0]:
# data column to date type convert
df["JoiningDate"] = pd.to_datetime(df["JoiningDate"])

In [0]:
df.head()

In [0]:
df.info()

In [0]:
df.describe()

# groupby().transform() / groupby().rank() /shift() / rolling() /expanding()
| SQL Window   | Pandas               |
| ------------ | -------------------- |
| ROW_NUMBER   | `groupby + cumcount` |
| RANK         | `rank()`             |
| AVG() OVER   | `transform("mean")`  |
| SUM() OVER   | `cumsum()`           |
| LAG          | `shift(1)`           |
| LEAD         | `shift(-1)`          |
| ROWS BETWEEN | `rolling()`          |



In [0]:
import pandas as pd

dfp = pd.DataFrame({
    "EmpID": [1,2,3,4,5,6,7,8,9,10],
    "EmpName": ["Alice","Bob","Charlie","Diana","Evan","Fiona","George","Hannah","Ian","Jane"],
    "Department": ["Sales","Sales","HR","HR","IT","IT","IT","Marketing","Marketing","Sales"],
    "Salary": [60000,55000,50000,52000,70000,72000,68000,48000,50000,58000],
    "JoiningDate": pd.to_datetime([
        "2020-03-01","2021-07-15","2019-02-10","2020-08-23",
        "2018-06-12","2019-11-30","2021-04-18","2020-01-20",
        "2021-05-22","2019-12-01"
    ]),
    "PerformanceRating": [5,4,3,5,2,4,5,3,4,3]
})


# Row Number 


In [0]:
dfp["row_number"]=(
    dfp.sort_values("Salary",ascending=False)\
    .groupby("Department") \
    .cumcount() + 1 \
    )

In [0]:
dfp

# Rank / Dense Rank 

In [0]:
dfp["rank"] =  df.groupby("Department")["Salary"]\
    .rank(method = "min" ,ascending = False )
dfp

In [0]:
dfp["dense_rank"] = df.groupby("Department")["Salary"] \
    .rank(method = "dense" , ascending= False )
dfp

# Aggregate Over (avg/sum/min/max)

In [0]:
#Department average salary
dfp["dept_Avg_Salary"] = df.groupby("Department")["Salary"].transform("mean")
dfp["dept_Avg_Salary"] 

In [0]:
dfp["diff_From_Avg"] = dfp["Salary"]-dfp["dept_Avg_Salary"]
dfp["diff_From_Avg"]

# RUNNING TOTAL (cumulative sum)

In [0]:
dfp=dfp.sort_values("JoiningDate")
df["running_Salary"] = df["Salary"].cumsum()

In [0]:
#Department wise running total
dfp["dept_running"] = (
    df.sort_values("JoiningDate")
    .groupby("Department")["Salary"]
    .cumsum()
)

In [0]:
dfp

# LAG/LEAD (previous / next row)


In [0]:
# Previous Salary
dfp["prev_salary"] = (
    dfp.sort_values(["Department","JoiningDate"])
    .groupby("Department")["Salary"]
    .shift(1)
)
dfp

In [0]:
dfp_sorted = dfp.sort_values(["Department","JoiningDate"])
dfp_sorted

In [0]:
dfp_sorted[[
    "Department",
    "EmpName",
    "JoiningDate",
    "Salary",
    "prev_salary"
]]


In [0]:
dfp["salary_change"]=dfp["Salary"] - dfp["prev_salary"]

In [0]:
dfp["moving_avg_2"] =(
    dfp.sort_values("JoiningDate")
    .groupby("Department")["Salary"]
    .rolling(2)
    .mean()
    .reset_index(level = 0 ,drop = True)
)
dfp

In [0]:
# Expanding window 
dfp["expanding_avg"] = (
    dfp.sort_values("JoiningDate")
    .groupby("Department")["Salary"]
    .expanding()
    .mean()
    .reset_index(level=0,drop=True)
)
dfp

In [0]:
dfp_sorted = dfp.sort_values(["Department","JoiningDate"])
dfp_sorted

Dataset: EmployeesPerformance

1. Rank employees based on Salary (highest salary = rank 1).
2. Dense Rank employees based on PerformanceRating (highest rating first).
3. Assign Row Numbers to employees ordered by JoiningDate.
4. Find the employee with the 2nd highest salary using RANK().
5. Find the employee with the 2nd highest salary using DENSE_RANK().
6. Find the 2nd most recent joiner using ROW_NUMBER().
7. Rank employees within each Department based on Salary.
8. Row Number employees within each Department ordered by JoiningDate.
9. Dense Rank employees within each Department based on PerformanceRating.
10. Find all employees who have the same salary rank.



In [0]:
# Rank employees based on Salary (highest salary = rank 1).
df["ranking"]=df["Salary"] \
                .rank(method="min" , ascending=False)
df

In [0]:
# Dense Rank employees based on PerformanceRating (highest rating first).
df["dense_Rank"]=df["PerformanceRating"] \
                .rank(method="dense" , ascending=False)
df

In [0]:
# Assign Row Numbers to employees ordered by JoiningDate.
# df["row_num"]=df.sort_values("JoiningDate") \
#         .reset_index(drop=True).index + 1
df = df.sort_values("JoiningDate")
df["row_num"] = df.index + 1


df

In [0]:
# Find the employee with the 2nd highest salary using RANK().
df["ranking"]=df["Salary"].rank(method="min",ascending=False)
second = df[df["ranking"] == 2]
second

In [0]:
#Find the employee with the 2nd highest salary using DENSE_RANK().
df["dense_ranking"] = df["Salary"].rank(method="dense" , ascending=False)
second_dense = df[df["dense_ranking"] == 2]
second_dense

In [0]:
#Find the 2nd most recent joiner using ROW_NUMBER().
df=df.sort_values("JoiningDate" , ascending=False)
df["row_numm"]=df.index+1
secondd=df[df["row_numm"] == 2]
secondd

In [0]:
#Rank employees within each Department based on Salary.
df["rdepart"]=df.groupby("Department")["Salary"].rank(
    method="min",ascending=False
)
df

In [0]:
#Row Number employees within each Department ordered by JoiningDate.
df = df.sort_values("JoiningDate")
df["rowN"]=(df.groupby("Department") \
    .cumcount() +1)
df

In [0]:
#Dense Rank employees within each Department based on PerformanceRating.
df["d_rank"]=df.groupby("Department")["PerformanceRating"].rank(method="dense" , ascending=False)
df

In [0]:
# Find all employees who have the same salary rank.
df["salary_rank"] = df["Salary"].rank(method="min",ascending=False)
duplicate_Rank = df["salary_rank"].duplicated(keep=False)
same_salary_rank = df[duplicate_Rank]
same_salary_rank

In [0]:
df["salary_rank"] = df["Salary"].rank(method="min",ascending=False)
same_Salary_Rank=df[df["salary_rank"].duplicated(keep=False)]
same_salary_rank

In [0]:
#11. Show each employee’s salary along with the next employee's salary (based on salary descending).
df_sorted=df.sort_values("Salary",ascending=False)
df_sorted["next_salary"]=df_sorted["Salary"].shift(-1)
df_sorted[["EmpName","Salary","next_salary"]]

In [0]:
# 12. Show each employee’s salary along with the previous employee’s salary.
df_pre_sort=df.sort_values("Salary", ascending=False)
df_pre_sort["pre_salary"]=df_pre_sort["Salary"].shift(1)
df_pre_sort[["EmpName","Salary","pre_salary"]]

In [0]:
# 13. Find the salary difference between an employee and the next employee.
df_sorted["difference"]=df_sorted["Salary"]-df_sorted["next_salary"]
df_sorted[["EmpName", "Salary", "next_salary", "difference"]]

In [0]:
# 14. Find the salary difference between an employee and the previous employee.
df_pre_sort["difference"]=df_pre_sort["Salary"]-df_pre_sort["pre_salary"]
df_pre_sort

In [0]:
# 15. For each department, show each employee and the next joining employee.

df_sorted = df.sort_values("JoiningDate")

df_sorted["next_employee"] = (
    df_sorted
    .groupby("Department")["EmpName"]
    .shift(-1)
)

df_sorted[["Department", "EmpName", "JoiningDate", "next_employee"]]
